In [46]:
import os

import psycopg
from azure.identity import DeviceCodeCredential


HOST = "pg-5104-distribucio.postgres.database.azure.com"
PORT = 5432
BASE_DE_DADES = "erp_distribucio"
SSLMODE = "require"          # el servidor només accepta connexions xifrades
TENANT = "politecnicllevant.cat"   # el directori Entra de l'institut
ESCOP = "https://ossrdbms-aad.database.windows.net/.default"   # el "recurs" de les bases de dades d'Azure

USUARI = "agalmes1796"
TOKEN = "pla-5u7sm-qjv4r"

print(USUARI)
print(TOKEN)


# def connecta(base_de_dades=BASE_DE_DADES):
#    return psycopg.connect(host=HOST, port=PORT, dbname=base_de_dades, user=USUARI,
#                           password=TOKEN, sslmode=SSLMODE, connect_timeout=15, autocommit=True, keepalives=1, keepalives_idle=30, keepalives_interval=10, keepalives_count=3)

def connecta(base_de_dades=BASE_DE_DADES):
    return psycopg.connect(
        host=HOST, port=PORT, dbname=base_de_dades, user=USUARI, password=TOKEN,
        sslmode=SSLMODE, connect_timeout=15,
        autocommit=True,
        keepalives=1, keepalives_idle=30, keepalives_interval=10, keepalives_count=3,
        tcp_user_timeout=30000,
        options="-c statement_timeout=30000",
    )
con = connecta()
print("Connectat com a", con.execute("SELECT current_user").fetchone()[0])
print(con.execute("SELECT version()").fetchone()[0])
print("Connexió xifrada:", con.info.pgconn.ssl_in_use)

agalmes1796
pla-5u7sm-qjv4r
Connectat com a agalmes1796
PostgreSQL 16.15 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 13.2.0, 64-bit
Connexió xifrada: True


In [2]:
import pandas as pd


def consulta(sql, params=None):
    """Executa una consulta i torna un DataFrame amb els noms de les columnes."""
    with con.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

In [3]:
consulta("""
    SELECT current_database(), current_user, current_schema();
""")

,current_database,current_user,current_schema
0,erp_distribucio,agalmes1796,erp


In [16]:
consulta("""
    SELECT table_schema, table_name 
FROM information_schema.tables 
WHERE table_type = 'BASE TABLE'
  AND table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;

    """)

,table_schema,table_name
0,erp,clients
1,erp,comandes
2,erp,comercials
3,erp,linies_comanda
4,erp,productes
5,erp,proveidors
6,erp,rutes
7,erp,vehicles


In [40]:
consulta("""
SELECT c.id, c.nom_comercial, ROUND(
SUM(lc.quantitat * lc.preu_unitari * (1 - lc.descompte_pct/100)),2) AS import_total
FROM clients c
JOIN comandes co ON c.id = co.client_id
JOIN linies_comanda lc ON lc.comanda_id = co.id
GROUP BY c.id, c.nom_comercial
ORDER BY import_total DESC
LIMIT 5
""")

,id,nom_comercial,import_total
0,309,Hotel Sa Cala 309,503968.84
1,266,Hostal Es Molí 266,480905.88
2,267,Hotel Tramuntana 267,465876.89
3,52,Hotel Sol 52,463845.33
4,384,Hotel Es Port 384,451852.77


In [44]:
consulta("""
SELECT c.id, c.nom_comercial, ROUND(
SUM(lc.quantitat * lc.preu_unitari * (1 - lc.descompte_pct/100)),2) AS import_total
FROM clients c
JOIN comandes co ON c.id = co.client_id
JOIN linies_comanda lc ON lc.comanda_id = co.id
WHERE co.estat <> 'ANUL·LADA'
GROUP BY c.id, c.nom_comercial
ORDER BY import_total DESC
LIMIT 5
""")

,id,nom_comercial,import_total
0,309,Hotel Sa Cala 309,495477.13
1,266,Hostal Es Molí 266,475087.55
2,52,Hotel Sol 52,461579.92
3,267,Hotel Tramuntana 267,457570.36
4,384,Hotel Es Port 384,444540.13


In [43]:
consulta("""
SELECT estat FROM comandes 
GROUP BY estat
""")

,estat
0,FACTURADA
1,PENDENT
2,SERVIDA
3,ANUL·LADA


In [30]:
consulta("""
SELECT * FROM clients LIMIT 1;
""")

,id,codi,nom_comercial,nif,tipus,municipi,codi_postal,zona,email,telefon,data_alta,actiu,comercial_id
0,1,C00001,Celler Sa Cala 1,D72251864,RESTAURANT,Palma,07001,PALMA,None,97163196364,2016-04-04,True,1


In [31]:
consulta("""
SELECT 
tipus, zona, 
count(*) FROM clients
GROUP BY tipus, zona
HAVING count(*) > 30
ORDER BY count(*) DESC
LIMIT 1
;
""")

,tipus,zona,count
0,HOTEL,PALMA,61


In [54]:
consulta("""
SELECT * FROM comandes LIMIT 1;
""")

,id,num_comanda,client_id,data_comanda,data_entrega,estat,canal,ruta_id,observacions
0,1,P23-0000001,21,2023-01-02 18:18:25+00:00,2023-01-03,FACTURADA,TELEFON,14,None


In [ ]:
consulta("""SELECT codi, nom comercial, municipi, import


ORDER BY count(*) DESC
LIMIT 5
    """)

In [67]:
consulta("""SELECT r.codi, 
    count(co.id) AS num_comandes,
    CONCAT(ROUND(SUM(lc.quantitat * lc.preu_unitari * (1 - lc.descompte_pct/100)),2), ' €') AS import_total,
    ROUND(AVG(lc.quantitat * lc.preu_unitari * (1 - lc.descompte_pct/100)),2) AS import_mitja
    FROM rutes r
    JOIN comandes co ON r.id = co.ruta_id
    JOIN linies_comanda lc ON lc.comanda_id = co.id
    WHERE data_comanda BETWEEN '2025-12-01' AND '2026-01-01'
    GROUP BY r.codi
    ORDER BY import_total DESC
    """)



,codi,num_comandes,import_total,import_mitja
0,R-LLE-2,348,91034.30 €,261.59
1,R-TRA-2,311,88293.25 €,283.90
2,R-PON-2,297,84331.46 €,283.94
3,R-TRA-1,278,83448.68 €,300.18
4,R-PAL-2,673,204230.26 €,303.46
5,R-NOR-1,592,199167.87 €,336.43
6,R-PLA-1,57,18877.60 €,331.19
7,R-NOR-2,524,182082.91 €,347.49
8,R-PLA-2,47,17345.87 €,369.06
9,R-PAL-1,457,147980.57 €,323.81
